In [ ]:
from config import setup_ai
import langchain_community
import pypdf
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
import chromadb

# 1. Re-connect to the existing folder (Automatic)
client = chromadb.PersistentClient(path="./my_vectordb")

# 2. Get the existing collection (Automatic)
# 'get_or_create' means: "If it exists, just open it. Don't make a new one."
collection = client.get_or_create_collection(name="user_knowledge")

print("Database is initialized and ready!")

In [ ]:
# Initialize the model using our central config
model = setup_ai()

In [ ]:
# Now you can use 'model' (which is the new Client)
if model:
    # Notice: we add '.models.' and specify the model name inside the function
    response = model.models.generate_content(
        model='gemini-2.5-flash', 
        contents="Give me a 1-sentence welcome for a new Data Engineer."
    )
    
    # The output is the same: we still use .text
    print(f"\n🤖 Gemini says: {response.text}")

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def prepare_data(file_path):
    """
    Loads a file (PDF or Text) and splits it into manageable chunks for the AI's long-term memory.
    """

    # 1. Determine file type and load the comtent
    if file_path.endswith('.pdf'):
        loader = PyPDFLoader(file_path)
    else:
        loader = TextLoader(file_path)
    
    # This reads the file and creates a list of 'Document' objects
    raw_documents = loader.load()

    # 2. Configure the text splitter
    # chunk_size: How many characters in each block
    # chunk_overlap: How many characters to repeat from the previous block
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 50,
        add_start_index=True,
        separators=["\n\n", "\n", " ", ""]
    )

    # 3. Perform the split
    chunks = text_splitter.split_documents(raw_documents)

    print(f"Data Audit: File split into {len(chunks)} chunks.")

    return chunks

# Example of how you will call this function:
# my_chunks = prepare_data("my_notes.txt")

In [ ]:
# 1. Run the function on test file
my_chunks = prepare_data("test_data.txt")

# 2. Check the "type" to ensure it returned a list
print(f"Data type: {type(my_chunks)}")

# 3. Look at the very first chunk to see if it looks right
if len(my_chunks) > 0:
    print("---First Chunk Content---")
    print(my_chunks[0].page_content)
    print("-------------------------")
    print(f"Metadata: {my_chunks[0].metadata}")

In [ ]:
# 1. Grab the text from the first chunk
text_sample = my_chunks[0].page_content

# 2. Using the client to get the math
response = model.models.embed_content(model='text-embedding-004', contents=text_sample)

In [ ]:
# Dig into the response to get actual numbers
vector = response.embeddings[0].values

print("Success! Your text is now math.")
print(f"First 5 numbers of the vector: {vector[:5]}")
print(f"Total length of the vector: {len(vector)}")

In [ ]:
query_text = "What should I eat for dinner?"

In [ ]:
query_vector = model.models.embed_content(model='text-embedding-004', contents=query_text)

In [ ]:
len(query_vector.embeddings[0].values)

In [ ]:
all_scores = []

In [ ]:
query_vector_values = query_vector.embeddings[0].values

for v in my_chunks:
    text_sample = v.page_content
    response = model.models.embed_content(model='text-embedding-004', contents=text_sample)
    vector = response.embeddings[0].values
    current_score = cosine_similarity([query_vector_values], [vector])[0][0]
    all_scores.append(current_score)

print("Results for each chunk:", all_scores)

In [ ]:
best_index = np.argmax(all_scores)
print(f"The AI found the best match for query: {query_text} and it is:")
print(my_chunks[best_index].page_content)

In [ ]:
my_vectors = []

for v in my_chunks:
    response = model.models.embed_content(model='text-embedding-004', contents=v.page_content)
    my_vectors.append(response.embeddings[0].values)


def find_similarity(query_text, chunks, vectors):
    query_vector = model.models.embed_content(model='text-embedding-004', contents=query_text)
    query_vector_values = query_vector.embeddings[0].values
    all_scores = []
    for v in vectors:
        current_score = cosine_similarity([query_vector_values], [v])[0][0]
        all_scores.append(current_score)

    best_index = np.argmax(all_scores)
    best_score = np.max(all_scores)
    if best_score >= 0.4:
        return f"The AI found the best match for query: {query_text} and it is: {my_chunks[best_index].page_content}"
    else:
        return "I'm sorry, I don't have information about that in my database."

In [ ]:
query_text_test = "What is the syntax for repeating a task in a programming language?"

In [ ]:
find_similarity(query_text_test, my_chunks, my_vectors)